
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning" style="width: 600px">
</div>


# 02L - Embeddings, Vector Databases, and Search


In this lab, we will apply the text vectorization, search, and question answering workflow that you learned in the demo. The dataset we will use this time will be on talk titles and sessions from [Data + AI Summit 2023](https://www.databricks.com/dataaisummit/). 

### ![Dolly](https://files.training.databricks.com/images/llm/dolly_small.png) Learning Objectives
1. Learn how to use Chroma to store your embedding vectors and conduct similarity search
1. Use OpenAI GPT-3.5 to generate response to your prompt

## Read data

In [1]:
import pandas as pd

dais_pdf = pd.read_parquet("dais23_talks.parquet")
display(dais_pdf)

,Title,Abstract
0,AI in Healthcare,This session will explore the transformative i...
1,Natural Language Processing: From Theory to Pr...,Dive into the world of Natural Language Proces...
2,Reinforcement Learning in Autonomous Systems,Discover how reinforcement learning techniques...
3,Ethical Considerations in AI Development,"As AI continues to evolve, ethical considerati..."
4,Deep Learning for Computer Vision,Explore the world of deep learning and compute...
5,AI-powered Fraud Detection,Fraud detection is a critical challenge for or...
6,Generative Adversarial Networks: Creating Real...,Learn how generative adversarial networks (GAN...
7,Explainable AI: Interpreting Black Box Models,Interpretability is a key factor in deploying ...
8,AI for Financial Services,This session will showcase how AI is revolutio...
9,Reinventing Customer Experience with AI,Discover how AI is reshaping the customer expe...


In [2]:
dais_pdf["full_text"] = dais_pdf.apply(
    lambda row: f"""Title: {row["Title"]}
                Abstract:  {row["Abstract"]}""".strip(),
    axis=1,
)
print(dais_pdf.iloc[0]["full_text"])

Title: AI in Healthcare
                Abstract:  This session will explore the transformative impact of artificial intelligence in the healthcare industry. Artificial intelligence (AI) has the potential to revolutionize healthcare by improving diagnostics, enabling personalized treatment plans, and enhancing patient monitoring. Join us as we delve into the latest advancements in medical imaging analysis, predictive analytics for disease management, and AI-powered clinical decision support systems. Discover how AI algorithms can analyze vast amounts of patient data, identify patterns, and provide valuable insights for healthcare professionals. Learn about the challenges and opportunities of implementing AI in healthcare and the potential for improving patient outcomes and reducing healthcare costs.


In [3]:
texts = dais_pdf["full_text"].to_list()

## Question 1
Set up Chroma and create collection

In [4]:
import chromadb

chroma_client = chromadb.Client()


Assign the value of `my_talks` to the `collection_name` variable.

In [5]:
# TODO
collection_name = "my_talks"

# If you have created the collection before, you need to delete the collection first
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
    chroma_client.delete_collection(name=collection_name)
else:
    print(f"Creating collection: '{collection_name}'")
    talks_collection = chroma_client.create_collection(name=collection_name)

Creating collection: 'my_talks'


## Question 2

[Add](https://docs.trychroma.com/reference/Collection#add) data to the collection. 

In [6]:
# TODO
talks_collection.add(
    documents=texts,
    ids=[f"id{x}" for x in range(len(texts))]
)

## Question 3

[Query](https://docs.trychroma.com/reference/Collection#query) for relevant documents. If you are looking for talks related to language models, your query texts could be `language models`. 

In [7]:
# TODO
import json

results = talks_collection.query(
    query_texts="language models",
    n_results=3
)

print(json.dumps(results, indent=4))

{
    "ids": [
        [
            "id1",
            "id7",
            "id9"
        ]
    ],
    "embeddings": null,
    "documents": [
        [
            "Title: Natural Language Processing: From Theory to Practice\n                Abstract:  Dive into the world of Natural Language Processing (NLP) as we explore the theoretical foundations and practical applications of NLP algorithms. NLP is a subfield of AI that focuses on enabling machines to understand and process human language. In this session, we will cover various NLP techniques, including text classification, sentiment analysis, named entity recognition, and machine translation. Learn about the underlying models and architectures used in NLP, such as recurrent neural networks (RNNs), transformers, and pre-trained language models like BERT. Gain practical insights into building NLP pipelines, leveraging existing libraries and frameworks, and applying NLP to real-world use cases across industries.",
            "Title: E

## Question 4

Load a language model and create a [pipeline](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines).

In [8]:
# TODO
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Pick a model from HuggingFace that can generate text
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)

pipe = pipeline(
    "text-generation", model=lm_model, tokenizer=tokenizer, max_new_tokens=512, device_map="auto", handle_long_generation="hole"
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Question 5

Prompt engineering for question answering

In [9]:
# TODO
# Come up with a question that you need the LLM assistant to help you with
# A sample question is "Help me find sessions related to XYZ" 
# Note: Your "XYZ" should be related to the query you passed in Question 3. 
question = "What can language models do?"

# Provide all returned similar documents from the cell above below
context = "\n".join(*results["documents"])

# Feel free to be creative how you construct the prompt. You can use the demo notebook as a jumpstart reference.
# You can also provide more requirements in the text how you want the answers to look like.
# Example requirement: "Recommend top-5 relevant sessions for me to attend."
prompt_template = f"Please answer the question ```{question}``` based on the following abstracts of AI summit:\n{context}"
print(prompt_template)

Please answer the question ```What can language models do?``` based on the following abstracts of AI summit:
Title: Natural Language Processing: From Theory to Practice
                Abstract:  Dive into the world of Natural Language Processing (NLP) as we explore the theoretical foundations and practical applications of NLP algorithms. NLP is a subfield of AI that focuses on enabling machines to understand and process human language. In this session, we will cover various NLP techniques, including text classification, sentiment analysis, named entity recognition, and machine translation. Learn about the underlying models and architectures used in NLP, such as recurrent neural networks (RNNs), transformers, and pre-trained language models like BERT. Gain practical insights into building NLP pipelines, leveraging existing libraries and frameworks, and applying NLP to real-world use cases across industries.
Title: Explainable AI: Interpreting Black Box Models
                Abstract: 

## Question 6 

Submit query for language model to generate response.

Hint: If you run into the error `index out of range in self`, make sure to check out this [documentation page](https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.TextGenerationPipeline.__call__.handle_long_generation).

In [10]:
# TODO
lm_response = pipe(prompt_template, max_new_tokens=512, truncation=True)
print(lm_response[0]["generated_text"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Please answer the question ```What can language models do?``` based on the following abstracts of AI summit:
Title: Natural Language Processing: From Theory to Practice
                Abstract:  Dive into the world of Natural Language Processing (NLP) as we explore the theoretical foundations and practical applications of NLP algorithms. NLP is a subfield of AI that focuses on enabling machines to understand and process human language. In this session, we will cover various NLP techniques, including text classification, sentiment analysis, named entity recognition, and machine translation. Learn about the underlying models and architectures used in NLP, such as recurrent neural networks (RNNs), transformers, and pre-trained language models like BERT. Gain practical insights into building NLP pipelines, leveraging existing libraries and frameworks, and applying NLP to real-world use cases across industries.
Title: Explainable AI: Interpreting Black Box Models
                Abstract: 

Notice that the output isn't exactly helpful. Head on to using OpenAI to try out GPT-3.5 instead! 

## OPTIONAL (Non-Graded): Use OpenAI models for Q/A

For this section to work, you need to generate an Open AI key. 

Steps:
1. You need to [create an account](https://platform.openai.com/signup) on OpenAI. 
2. Generate an OpenAI [API key here](https://platform.openai.com/account/api-keys). 

Note: OpenAI does not have a free option, but it gives you $5 as credit. Once you have exhausted your $5 credit, you will need to add your payment method. You will be [charged per token usage](https://openai.com/pricing). **IMPORTANT**: It's crucial that you keep your OpenAI API key to yourself. If others have access to your OpenAI key, they will be able to charge their usage to your account! 

In [ ]:
# TODO
import os

os.environ["OPENAI_API_KEY"] = "<FILL IN>"

In [ ]:
import openai

openai.api_key = os.environ["OPENAI_API_KEY"]

If you would like to estimate how much it would cost to use OpenAI, you can use `tiktoken` library from OpenAI to get the number of tokens from your prompt.


We will be using `gpt-3.5-turbo` since it's the most economical option at ($0.002/1k tokens), as of May 2023. GPT-4 charges $0.04/1k tokens. The following code block below is referenced from OpenAI's documentation on ["Managing tokens"](https://platform.openai.com/docs/guides/chat/managing-tokens).

In [ ]:
import tiktoken

price_token = 0.002
encoder = tiktoken.encoding_for_model("gpt-3.5-turbo")
cost_to_run = len(encoder.encode(prompt_template)) / 1000 * price_token
print(f"It would take roughly ${round(cost_to_run, 5)} to run this prompt")

We won't have to create a new vector database again. We can just send our `context` from above to OpenAI. We will use their chat completion API to interact with `GPT-3.5-turbo`. You can refer to their [documentation here](https://platform.openai.com/docs/guides/chat).

Something interesting is that OpenAI models use the system message to help their assistant to be more accurate. From OpenAI's [docs](https://platform.openai.com/docs/guides/chat/introduction):

> Future models will be trained to pay stronger attention to system messages. The system message helps set the behavior of the assistant.



In [0]:
# TODO
gpt35_response = openai.ChatCompletion.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": <FILL_IN>},
    ],
    temperature=0, # 0 makes outputs deterministic; The closer the value is to 1, the more random the outputs are for each time you re-run.
)

In [0]:
print(gpt35_response.choices[0]["message"]["content"])

In [0]:
from IPython.display import Markdown

Markdown(gpt35_response.choices[0]["message"]["content"])

We can also check how many tokens OpenAI has used

In [0]:
gpt35_response["usage"]["total_tokens"]

The results are noticeably much better compared to when using Hugging Face's GPT-2! It didn't get stuck in the text generation, but the sessions recommended are not all relevant to pandas either. You can further do more prompt engineering to get better results.

## Submit your Results (edX Verified Only)

To get credit for this lab, click the submit button in the top right to report the results. If you run into any issues, click `Run` -> `Clear state and run all`, and make sure all tests have passed before re-submitting. If you accidentally deleted any tests, take a look at the notebook's version history to recover them or reload the notebooks.

&copy; 2023 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/>
<a href="https://databricks.com/privacy-policy">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use">Terms of Use</a> | <a href="https://help.databricks.com/">Support</a>